# GroupBy Object in Pandas — Session 19

In earlier sessions you learned to build, inspect, filter, and clean Series and DataFrames. Everything you did there operated on the whole table at once. Real questions, however, are almost never about the whole table — they are about its parts. Who is the top earner among action movies? Which director gets the most votes on average? How many movies did each actor lead? Every one of these questions splits the data into groups, computes something inside each group, and joins the answers back together. That pattern is so fundamental that pandas ships a dedicated engine for it: the GroupBy object, built on the split-apply-combine strategy.

Split-apply-combine is the mental model for this whole session. SPLIT: pandas divides the rows according to the distinct values of one or more columns — a group per genre, per director, per actor. APPLY: a function runs inside each group — sum, mean, count, or any function you write yourself. COMBINE: all the per-group results are collected back into one Series or DataFrame, indexed by the group keys. If you have ever used SQL's GROUP BY or a pivot table in Excel, you already know this concept; this session gives you the pandas spelling of it.

The session takes you through the GroupBy toolkit step by step. You start by creating a GroupBy object and confirming that it is lazy — grouping alone does nothing until you apply an operation. Then you run the built-in aggregations: sum, mean, std, and count across every group, chained to sort_values and head for leaderboards. Next come the attributes that inspect the grouping itself: len and size for group counts, get_group to pull out one group, groups to see the row-index mapping, and describe, nunique, sample, and nth for group-wise summaries.

From there the session moves to richer tools. The agg method runs several different aggregations in a single call using a dictionary or a list. Looping shows how to iterate (group_name, group_dataframe) pairs manually. apply is the full-power escape hatch that runs any custom function on every group — used here for within-group rankings and normalisation. Finally, GroupBy on multiple columns builds nested groups whose keys are tuples and whose results carry a MultiIndex. The notebook closes with a cricket exercise on ball-by-ball deliveries, where all these patterns come together: most runs, most sixes, death-over hitters, and a reusable function for any batsman's highest score.

Work every code cell in order. GroupBy is one of those tools you can only master by running — predict the output first, then check it against the line below.

# 1. Setup — Imports and Loading the Data

## 1.1 The code that prepares the session

Every analysis begins the same way — import the libraries and load the file.

```python
import pandas as pd
import numpy as np

movies = pd.read_csv('/content/imdb-top-1000.csv')
movies.head()
```

`import pandas as pd` and `import numpy as np` bring in the two pillars of the course; numpy is imported because some GroupBy operations return numpy-style results. `pd.read_csv('/content/imdb-top-1000.csv')` reads the IMDb top-1000 films file (the path assumes the file exists in your Colab environment), and `movies.head()` shows the first five rows.

The head output defines the columns you will group by for the rest of the session: `Series_Title`, `Released_Year`, `Runtime`, `Genre`, `IMDB_Rating`, `Director`, `Star1`, `Star2`, `Star3`, `Star4`, `No_of_Votes`, `Gross`, and `Metascore`. Notice that `Genre` is a single value per movie (Action, Drama, Comedy, and so on) and `Star1` is the lead actor — both are ideal grouping keys. `Gross` is the box-office total, `No_of_Votes` the popularity proxy, and `IMDB_Rating` the score.

Hold one fact from the output as a reference point: The Shawshank Redemption heads the file with a 9.3 rating. When the session later asks for the highest-rated movie per genre, you will already know which answer to expect for Drama.

In [166]:
import pandas as pd
import numpy as np

In [167]:
movies = pd.read_csv('/content/imdb-top-1000.csv')

In [170]:
movies.head()

               Series_Title Released_Year  Runtime   Genre  IMDB_Rating  \
0  The Shawshank Redemption          1994      142   Drama          9.3   
1             The Godfather          1972      175   Crime          9.2   
2           The Dark Knight          2008      152  Action          9.0   
3    The Godfather: Part II          1974      202   Crime          9.0   
4              12 Angry Men          1957       96   Crime          9.0   

               Director           Star1  No_of_Votes        Gross  Metascore  
0        Frank Darabont     Tim Robbins      2343110   28341469.0       80.0  
1  Francis Ford Coppola   Marlon Brando      1620367  134966411.0      100.0  
2     Christopher Nolan  Christian Bale      2303232  534858444.0       84.0  
3  Francis Ford Coppola       Al Pacino      1129952   57300000.0       90.0  
4          Sidney Lumet     Henry Fonda       689845    4360000.0       96.0  

# 2. GroupBy — Split, Apply, Combine

## 2.1 Grouping is step zero

```python
genres = movies.groupby('Genre')
```

A single line — and nothing appears to happen, because nothing has happened yet. `movies.groupby('Genre')` creates a GroupBy object: it has registered 'make a per-Genre group out of every row', but it has computed nothing. The object is lazy. Only when you hang an operation off it — `.sum()`, `.mean()`, `.agg()`, `.apply()` — does pandas actually split the frame, run the function on each group, and combine the results.

The idea can be captured as a slogan: make small groups, work on each group, join them back. This is the split-apply-combine strategy. When you write `movies.groupby('Genre')['Gross'].sum()`, pandas splits the frame by genre, applies `sum` to the Gross column within every group, and combines the totals into one Series whose index is the genre names — that is the combine step producing the final result.

Grouping keys become the output index. Since groups are named by their key values, the result of a group-aggregate is always 'answer indexed by category'. And because the GroupBy object is cheap to create, you can store it in a variable (`genres`) once and hang many different operations off it — which is exactly what the next several cells do.

> Tip: compare with SQL — `SELECT genre, SUM(gross) FROM movies GROUP BY genre` — or with an Excel pivot table. Same idea, pandas spelling.

In [172]:
genres = movies.groupby('Genre')

# 3. Built-in Aggregations

## 3.1 Aggregating a GroupBy

The simplest way to make a lazy GroupBy object do work is to call one of pandas' built-in aggregation methods on it.

```python
genres.std()                                   # std of every numeric column, per group
movies.groupby('Genre').sum()['Gross']         # per-genre Gross sum
movies.groupby('Genre')['IMDB_Rating'].mean()  # per-genre average rating
```

`genres.std()` computes the standard deviation of EVERY numeric column separately within each group — Runtime, IMDB_Rating, No_of_Votes, Gross, Metascore — and returns a DataFrame whose rows are genres and whose columns are those statistics. Sum, mean, count, min, max, and median behave the same way: pick a function, apply it to every group.

The column selection trick `['Gross']` narrows the aggregation to one column. `movies.groupby('Genre').sum()['Gross']` first sums everything, then keeps the Gross column; the shorter and faster equivalent `movies.groupby('Genre')['Gross'].sum()` selects the column first and only sums that. Both compile to the same answer.

Chaining makes leaderboards trivial: `movies.groupby('Genre')['Gross'].sum().sort_values(ascending=False).head(3)` answers 'top three genres by total earnings' (Drama, then Action, then Comedy), and `movies.groupby('Genre')['IMDB_Rating'].mean().sort_values(ascending=False).head(1)` finds the highest-rated genre (Western at 8.35). The `count()` idiom counts rows per group: `movies.groupby('Star1')['Series_Title'].count()` answers 'how many movies did each actor lead' — Tom Hanks at the top with twelve. GroupBy plus aggregate plus sort is the entire recipe for every 'top N by category' question.

In [178]:
# Applying builtin aggregation fuctions on groupby objects
genres.std()

             Runtime  IMDB_Rating    No_of_Votes         Gross  Metascore
Genre                                                                    
Action     28.500706     0.304258  432946.814748  2.256724e+08  12.421252
Adventure  33.317320     0.229781  301188.347642  1.697543e+08  12.345393
Animation  14.530471     0.253221  262173.231571  2.091840e+08   8.813646
Biography  25.514466     0.267140  271284.191372  1.363251e+08  11.028187
Comedy     22.946213     0.228771  188653.570564  1.946513e+08  11.829160
Crime      27.689231     0.335477  373999.730656  1.571191e+08  13.099102
Drama      27.740490     0.267229  305554.162841  2.201164e+08  12.744687
Family     10.606602     0.000000  137008.302816  3.048412e+08  16.970563
Fantasy    12.727922     0.141421   22179.111299  7.606861e+07        NaN
Film-Noir   4.000000     0.152753   54649.083277  7.048472e+07   1.527525
Horror     13.604812     0.311302  234883.508691  9.965017e+07  15.362291
Mystery    14.475423     0.310791  404

In [184]:
# find the top 3 genres by total earning
movies.groupby('Genre').sum()['Gross'].sort_values(ascending=False).head(3)

Genre
Drama     3.540997e+10
Action    3.263226e+10
Comedy    1.566387e+10
Name: Gross, dtype: float64

In [187]:
movies.groupby('Genre')['Gross'].sum().sort_values(ascending=False).head(3)

Genre
Drama     3.540997e+10
Action    3.263226e+10
Comedy    1.566387e+10
Name: Gross, dtype: float64

In [190]:
# find the genre with highest avg IMDB rating
movies.groupby('Genre')['IMDB_Rating'].mean().sort_values(ascending=False).head(1)

Genre
Western    8.35
Name: IMDB_Rating, dtype: float64

In [192]:
# find director with most popularity
movies.groupby('Director')['No_of_Votes'].sum().sort_values(ascending=False).head(1)

Director
Christopher Nolan    11578345
Name: No_of_Votes, dtype: int64

In [193]:
# find the highest rated movie of each genre
# movies.groupby('Genre')['IMDB_Rating'].max()

Genre
Action       9.0
Adventure    8.6
Animation    8.6
Biography    8.9
Comedy       8.6
Crime        9.2
Drama        9.3
Family       7.8
Fantasy      8.1
Film-Noir    8.1
Horror       8.5
Mystery      8.4
Thriller     7.8
Western      8.8
Name: IMDB_Rating, dtype: float64

In [196]:
# find number of movies done by each actor
# movies['Star1'].value_counts()

movies.groupby('Star1')['Series_Title'].count().sort_values(ascending=False)

Star1
Tom Hanks             12
Robert De Niro        11
Clint Eastwood        10
Al Pacino             10
Leonardo DiCaprio      9
                      ..
Glen Hansard           1
Giuseppe Battiston     1
Giulietta Masina       1
Gerardo Taracena       1
Ömer Faruk Sorak       1
Name: Series_Title, Length: 660, dtype: int64

# 4. GroupBy Attributes and Methods

## 4.1 Inspecting the groups themselves

Before asking what the groups contain, ask about the groups themselves. The GroupBy object exposes a small set of attributes and methods that describe its own structure.

`len(movies.groupby('Genre'))` returns 14 — the number of distinct groups — and `movies['Genre'].nunique()` independently confirms 14 distinct values in the column. `movies.groupby('Genre').size()` counts the rows in EVERY group, giving a full frequency table: Drama 289, Action 172, Comedy 155, Crime 107, and so on down to Thriller with a lonely 1. `size()` is exactly what `value_counts()` reports when called on the single grouping column.

`genres.nth(6)` returns the row at position 6 within each genre — the seventh movie in that genre's internal order — while the commented `first()` and `last()` would take the first or last row of each group. `genres.describe()` runs the whole statistical summary (count, mean, std, min, quartiles, max) for every numeric column, group by group, producing a large table of per-genre health statistics.

> Tip: `size()` counts rows per group, `nunique()` counts distinct values per group per column, `nth(k)` grabs one row. Learn each with the notebook's real outputs.

## 4.2 Deep dive — one group at a time

Two GroupBy features deserve a closer look because they pop up constantly: `get_group` and `groups`.

```python
genres.get_group('Fantasy')
```

`get_group('Fantasy')` returns the FULL DataFrame of one group — every movie whose Genre is Fantasy. The output shows exactly two rows: 'Das Cabinet des Dr. Caligari' (1920) and 'Nosferatu' (1922), both classic silent films. The equivalent filter `movies[movies['Genre'] == 'Fantasy']` produces the same two rows; `get_group` just gives the group's data by name. When you need to eyeball a single category before aggregating everyone, this is the tool.

```python
genres.groups
```

`groups` is a dictionary mapping every group key to the list of its row indices: `{'Action': [2, 5, 8, 10, ...], 'Drama': [...], ...}`. It is a map of the whole grouping in one object — useful to check which rows belong where without touching the data itself. It also explains why the GroupBy object feels like a folder of smaller DataFrames: one sub-DataFrame per key, each reachable by name.

Two sampling helpers round out the toolkit. `genres.sample(2, replace=True)` picks two random rows from every genre (with replacement), handy for building a balanced random sample from each category. And `genres.nunique()` counts, per column, how many DISTINCT values appear inside each group — for Action, 172 distinct movie titles, 61 release years, 15 distinct ratings, 123 directors. When someone asks 'how varied are the groups?', nunique is the one-word answer.

In [ ]:
# GroupBy Attributes and Methods
# find total number of groups -> len
# find items in each group -> size
# first()/last() -> nth item
# get_group -> vs filtering
# groups
# describe
# sample
# nunique

In [198]:
len(movies.groupby('Genre'))

14

In [199]:
movies['Genre'].nunique()

14

In [200]:
movies.groupby('Genre').size()

Genre
Action       172
Adventure     72
Animation     82
Biography     88
Comedy       155
Crime        107
Drama        289
Family         2
Fantasy        2
Film-Noir      3
Horror        11
Mystery       12
Thriller       1
Western        4
dtype: int64

In [204]:
genres = movies.groupby('Genre')
# genres.first()
# genres.last()
genres.nth(6)

                                             Series_Title Released_Year  \
Genre                                                                     
Action     Star Wars: Episode V - The Empire Strikes Back          1980   
Adventure                              North by Northwest          1959   
Animation                                          WALL·E          2008   
Biography                                      Braveheart          1995   
Comedy                                 The Great Dictator          1940   
Crime                                               Se7en          1995   
Drama                               It's a Wonderful Life          1946   
Horror                                            Get Out          2017   
Mystery                                            Sleuth          1972   

           Runtime  IMDB_Rating              Director             Star1  \
Genre                                                                     
Action         124      

In [201]:
movies['Genre'].value_counts()

Drama        289
Action       172
Comedy       155
Crime        107
Biography     88
Animation     82
Adventure     72
Mystery       12
Horror        11
Western        4
Film-Noir      3
Fantasy        2
Family         2
Thriller       1
Name: Genre, dtype: int64

In [209]:
genres.get_group('Fantasy')

movies[movies['Genre'] == 'Fantasy']

                     Series_Title Released_Year  Runtime    Genre  \
321  Das Cabinet des Dr. Caligari          1920       76  Fantasy   
568                     Nosferatu          1922       94  Fantasy   

     IMDB_Rating      Director          Star1  No_of_Votes        Gross  \
321          8.1  Robert Wiene  Werner Krauss        57428  337574718.0   
568          7.9   F.W. Murnau    Max Schreck        88794  445151978.0   

     Metascore  
321        NaN  
568        NaN  

In [210]:
genres.groups

{'Action': [2, 5, 8, 10, 13, 14, 16, 29, 30, 31, 39, 42, 44, 55, 57, 59, 60, 63, 68, 72, 106, 109, 129, 130, 134, 140, 142, 144, 152, 155, 160, 161, 166, 168, 171, 172, 177, 181, 194, 201, 202, 216, 217, 223, 224, 236, 241, 262, 275, 294, 308, 320, 325, 326, 331, 337, 339, 340, 343, 345, 348, 351, 353, 356, 357, 362, 368, 369, 375, 376, 390, 410, 431, 436, 473, 477, 479, 482, 488, 493, 496, 502, 507, 511, 532, 535, 540, 543, 564, 569, 570, 573, 577, 582, 583, 602, 605, 608, 615, 623, ...], 'Adventure': [21, 47, 93, 110, 114, 116, 118, 137, 178, 179, 191, 193, 209, 226, 231, 247, 267, 273, 281, 300, 301, 304, 306, 323, 329, 361, 366, 377, 402, 406, 415, 426, 458, 470, 497, 498, 506, 513, 514, 537, 549, 552, 553, 566, 576, 604, 609, 618, 638, 647, 675, 681, 686, 692, 711, 713, 739, 755, 781, 797, 798, 851, 873, 884, 912, 919, 947, 957, 964, 966, 984, 991], 'Animation': [23, 43, 46, 56, 58, 61, 66, 70, 101, 135, 146, 151, 158, 170, 197, 205, 211, 213, 219, 229, 230, 242, 245, 246, 270, 33

In [211]:
genres.describe()

          Runtime                                                              \
            count        mean        std    min     25%    50%     75%    max   
Genre                                                                           
Action      172.0  129.046512  28.500706   45.0  110.75  127.5  143.25  321.0   
Adventure    72.0  134.111111  33.317320   88.0  109.00  127.0  149.00  228.0   
Animation    82.0   99.585366  14.530471   71.0   90.00   99.5  106.75  137.0   
Biography    88.0  136.022727  25.514466   93.0  120.00  129.0  146.25  209.0   
Comedy      155.0  112.129032  22.946213   68.0   96.00  106.0  124.50  188.0   
Crime       107.0  126.392523  27.689231   80.0  106.50  122.0  141.50  229.0   
Drama       289.0  124.737024  27.740490   64.0  105.00  121.0  137.00  242.0   
Family        2.0  107.500000  10.606602  100.0  103.75  107.5  111.25  115.0   
Fantasy       2.0   85.000000  12.727922   76.0   80.50   85.0   89.50   94.0   
Film-Noir     3.0  104.00000

In [214]:
genres.sample(2,replace=True)

                            Series_Title Released_Year  Runtime      Genre  \
944                      Batoru rowaiaru          2000      114     Action   
625                           Apocalypto          2006      139     Action   
991                       Kelly's Heroes          1970      144  Adventure   
300                              Ben-Hur          1959      212  Adventure   
891                        Incredibles 2          2018      118  Animation   
389                       The Iron Giant          1999       86  Animation   
536              All the President's Men          1976      138  Biography   
635                        Walk the Line          2005      136  Biography   
826                          Barton Fink          1991      116     Comedy   
732       Me and Earl and the Dying Girl          2015      105     Comedy   
438                        Touch of Evil          1958       95      Crime   
222                            Prisoners          2013      153 

In [215]:
genres.nunique()

           Series_Title  Released_Year  Runtime  IMDB_Rating  Director  Star1  \
Genre                                                                           
Action              172             61       78           15       123    121   
Adventure            72             49       58           10        59     59   
Animation            82             35       41           11        51     77   
Biography            88             44       56           13        76     72   
Comedy              155             72       70           11       113    133   
Crime               106             56       65           14        86     85   
Drama               289             83       95           14       211    250   
Family                2              2        2            1         2      2   
Fantasy               2              2        2            2         2      2   
Film-Noir             3              3        3            3         3      3   
Horror               11     

# 5. agg — One Call, Many Aggregations

## 5.1 Per-column statistics in a single method

So far one aggregation at a time: sum, then mean, then min. Real reports rarely want just one. The `agg` method packages several aggregations into one call — the Swiss-army knife of group-wise reporting.

```python
genres.agg({'Runtime': 'mean', 'Gross': ['min', 'max', 'mean']})
genres.agg(['min', 'max', 'mean', 'sum'])
```

There are two syntaxes. A DICTIONARY picks functions per column: `{'Runtime': 'mean', ...}` says 'give me the mean Runtime per group', and a value that is itself a list, like `['min', 'max', 'mean']`, runs several functions on that one column. The dictionary syntax is the precise one — you choose exactly what happens to each column.

A LIST of function names applies the SAME list to every column: `genres.agg(['min', 'max', 'mean', 'sum'])` produces a wide DataFrame where each original column expands into four sub-columns. Function names are given as strings ('mean', 'sum', 'min', 'max', 'median', 'std'), but any callable — even `lambda x: x.max() - x.min()` — is accepted, so agg can also run your own per-group logic.

The notebook's cells combine both ideas: `genres.agg({'Runtime': ['min', 'mean'], 'IMDB_Rating': 'mean', 'No_of_Votes': ['sum', 'max'], 'Gross': 'sum', 'Metascore': 'min'})` gives a report card with exactly the statistics each column wants. When one aggregation per column feels limiting, reach for agg.

## 5.2 Deep dive — reading agg's output

The shape of agg's result tells you which syntax you used. With the dictionary form, the output keeps one row per group (Action, Adventure, Animation, ...) and one column per requested statistic — Runtime mean, IMDB_Rating mean, No_of_Votes sum, Gross sum, Metascore min all appear side by side. Reading the Action row: 129.05 mean runtime, 7.95 mean rating, 72.28 million votes in total, 3.26e10 total gross, and a minimum Metascore of 33. Action is the genre that earned most at the box office among its 172 films.

With the list form, `genres.agg(['min', 'max', 'mean', 'sum'])`, every numeric column fans out into four sub-columns and the column header becomes a TWO-LEVEL MultiIndex — `Runtime` over its four functions, then `IMDB_Rating` over the same four, and so on. That is why the printed frame looks far wider than before: 5 numeric columns times 4 functions.

The mixed form produces both effects at once — the Runtime and No_of_Votes headers become multi-level (two functions each), while IMDB_Rating, Gross, and Metascore stay single-level (one function each).

Two habits help you navigate these frames. Remember that the rows are always the group keys (the genres), and the columns are the aggregation results. And when you need to turn this summary back into a plain flat table, `reset_index()` flattens the group-key index into an ordinary column — a pattern you will use again with GroupBy results throughout the course.

In [217]:
# agg method
# passing dict
genres.agg(
    {
        'Runtime':'mean',
        'IMDB_Rating':'mean',
        'No_of_Votes':'sum',
        'Gross':'sum',
        'Metascore':'min'
    }
)

              Runtime  IMDB_Rating  No_of_Votes         Gross  Metascore
Genre                                                                   
Action     129.046512     7.949419     72282412  3.263226e+10       33.0
Adventure  134.111111     7.937500     22576163  9.496922e+09       41.0
Animation   99.585366     7.930488     21978630  1.463147e+10       61.0
Biography  136.022727     7.938636     24006844  8.276358e+09       48.0
Comedy     112.129032     7.901290     27620327  1.566387e+10       45.0
Crime      126.392523     8.016822     33533615  8.452632e+09       47.0
Drama      124.737024     7.957439     61367304  3.540997e+10       28.0
Family     107.500000     7.800000       551221  4.391106e+08       67.0
Fantasy     85.000000     8.000000       146222  7.827267e+08        NaN
Film-Noir  104.000000     7.966667       367215  1.259105e+08       94.0
Horror     102.090909     7.909091      3742556  1.034649e+09       46.0
Mystery    119.083333     7.975000      4203004  1.

In [219]:
# passing list
genres.agg(['min','max','mean','sum'])

          Runtime                         IMDB_Rating                         \
              min  max        mean    sum         min  max      mean     sum   
Genre                                                                          
Action         45  321  129.046512  22196         7.6  9.0  7.949419  1367.3   
Adventure      88  228  134.111111   9656         7.6  8.6  7.937500   571.5   
Animation      71  137   99.585366   8166         7.6  8.6  7.930488   650.3   
Biography      93  209  136.022727  11970         7.6  8.9  7.938636   698.6   
Comedy         68  188  112.129032  17380         7.6  8.6  7.901290  1224.7   
Crime          80  229  126.392523  13524         7.6  9.2  8.016822   857.8   
Drama          64  242  124.737024  36049         7.6  9.3  7.957439  2299.7   
Family        100  115  107.500000    215         7.8  7.8  7.800000    15.6   
Fantasy        76   94   85.000000    170         7.9  8.1  8.000000    16.0   
Film-Noir     100  108  104.000000    31

In [220]:
# Adding both the syntax
genres.agg(
    {
        'Runtime':['min','mean'],
        'IMDB_Rating':'mean',
        'No_of_Votes':['sum','max'],
        'Gross':'sum',
        'Metascore':'min'
    }
)

          Runtime             IMDB_Rating No_of_Votes                  Gross  \
              min        mean        mean         sum      max           sum   
Genre                                                                          
Action         45  129.046512    7.949419    72282412  2303232  3.263226e+10   
Adventure      88  134.111111    7.937500    22576163  1512360  9.496922e+09   
Animation      71   99.585366    7.930488    21978630   999790  1.463147e+10   
Biography      93  136.022727    7.938636    24006844  1213505  8.276358e+09   
Comedy         68  112.129032    7.901290    27620327   939631  1.566387e+10   
Crime          80  126.392523    8.016822    33533615  1826188  8.452632e+09   
Drama          64  124.737024    7.957439    61367304  2343110  3.540997e+10   
Family        100  107.500000    7.800000      551221   372490  4.391106e+08   
Fantasy        76   85.000000    8.000000      146222    88794  7.827267e+08   
Film-Noir     100  104.000000    7.96666

# 6. Looping Over Groups

## 6.1 The (group, dataframe) pair

A GroupBy object is iterable: looping over it yields one tuple per group, `(group_name, group_data)`. The single most useful pattern for manual inspection:

```python
for group, data in genres:
    print(group, data.shape)
```

Each iteration unpacks the group key (the genre, like 'Action') and the full DataFrame of that group's rows. This is the explicit form of split-apply-combine — you do the apply and combine yourself, by hand, instead of delegating to sum() or mean(). It is your debugging window onto what the groups actually contain.

Iteration is usually NOT the first tool you should reach for: built-in aggregations and agg are vectorised and far faster, and the session's apply method (next section) is more idiomatic for custom per-group logic. But looping wins in two situations: when you need to print or log something about each group, and when the group-specific logic is complex enough that forcing it into an agg dictionary or an apply function would obscure the code. Use the loop to explore and to build results manually, then move to the vectorised tool for production.

## 6.2 Deep dive — building a result manually

The notebook's loop demonstrates the manual combine step with a real goal:

```python
df = pd.DataFrame(columns=movies.columns)
for group, data in genres:
    df = df.append(data[data['IMDB_Rating'] == data['IMDB_Rating'].max()])

df
```

An empty DataFrame is created with `columns=movies.columns` to reserve the same column names. Inside the loop, for every genre, `data['IMDB_Rating'].max()` finds the genre's highest rating, the boolean mask `data['IMDB_Rating'] == ...` keeps only the movie(s) at that rating, and `append` stacks that winner onto the growing answer frame. The final `df` is the highest-rated movie of every genre — The Dark Knight for Action, Interstellar for Adventure, Spirited Away for Animation.

Two things to notice. First, the loop-style combine is explicit: you build the accumulator yourself instead of letting pandas do it. `df.append` is the pre-1.4 spelling of pd.concat, and appending inside a loop is fine for small passes (modern pandas prefers `pd.concat` on a list of frames). Second, when a genre's top rating is shared, the mask keeps EVERY tie — the same logic a groupby + max would flatten. For strictly one winner per genre, `drop_duplicates(subset='Genre')` or `.idxmax()`-based selection would be the next refinement. The loop remains the most readable way to see exactly what the combine step does.

In [227]:
# looping on groups
df = pd.DataFrame(columns=movies.columns)
for group,data in genres:
  df = df.append(data[data['IMDB_Rating'] == data['IMDB_Rating'].max()])

df

                            Series_Title Released_Year Runtime      Genre  \
2                        The Dark Knight          2008     152     Action   
21                          Interstellar          2014     169  Adventure   
23         Sen to Chihiro no kamikakushi          2001     125  Animation   
7                       Schindler's List          1993     195  Biography   
19                          Gisaengchung          2019     132     Comedy   
26                       La vita è bella          1997     116     Comedy   
1                          The Godfather          1972     175      Crime   
0               The Shawshank Redemption          1994     142      Drama   
688           E.T. the Extra-Terrestrial          1982     115     Family   
698  Willy Wonka & the Chocolate Factory          1971     100     Family   
321         Das Cabinet des Dr. Caligari          1920      76    Fantasy   
309                        The Third Man          1949     104  Film-Noir   

# 7. apply — Custom Functions

## 7.1 The custom escape hatch

Every aggregation so far used a built-in: sum, mean, or min. But split-apply-combine also needs CUSTOM logic — a statistic that does not exist, or a transformation that ranks movies inside their genre before combining. That door is `apply`.

```python
genres.apply(min)              # built-in function applied to each group

def foo(group):
    return group['Series_Title'].str.startswith('A').sum()
genres.apply(foo)              # a function you defined

def rank_movie(group):
    group['genre_rank'] = group['IMDB_Rating'].rank(ascending=False)
    return group
genres.apply(rank_movie)       # a function that transforms the group itself
```

`apply(fn)` passes each group's sub-DataFrame (or sub-Series) to the function and collects whatever it returns. When the function returns one value per group, the result is a Series indexed by group key — an aggregation. When the function returns a whole DataFrame (for example a ranked copy of the group), the results are stacked back into one big DataFrame — a transformation.

The notebook starts with `genres.apply(min)` — the built-in `min` run per group, which returns the 'smallest' row of each group. Then it moves to named functions: foo counts titles, rank_movie ranks inside the genre, and normal normalises ratings per group. This is the real 'apply' of split-apply-combine: arbitrary Python, executed per group, results combined. It is slower than built-ins — every group costs a Python function call — so prefer vectorised aggregations whenever the built-in expresses your question, and keep apply for the genuinely custom cases.

## 7.2 Deep dive — the three named functions

The workbook defines three functions and applies them, so read each one inside-out.

```python
def foo(group):
    return group['Series_Title'].str.startswith('A').sum()
```

`foo` counts, per genre, how many titles begin with A. `group['Series_Title'].str.startswith('A')` returns a boolean Series, and `.sum()` counts the True values — because True counts as 1. Applied, the result is a Series of integer counts indexed by genre: Drama 21, Comedy 14, Action 10, and 0 for the small genres Family, Fantasy, and Mystery.

```python
def rank_movie(group):
    group['genre_rank'] = group['IMDB_Rating'].rank(ascending=False)
    return group
```

`rank_movie` adds a column. Inside the group, `rank(ascending=False)` gives the best-rated movie rank 1; every movie of that genre carries its own genre_rank, so the returned group now has an extra column. Because the function returns a whole DataFrame, `genres.apply(rank_movie)` returns the ENTIRE movies table with the new genre_rank column — a transformation, not a reduction.

```python
def normal(group):
    group['norm_rating'] = (group['IMDB_Rating'] - group['IMDB_Rating'].min()) / (group['IMDB_Rating'].max() - group['IMDB_Rating'].min())
    return group
```

`normal` min-max scales the rating within each genre: the genre's best movie becomes 1.0, its worst 0.0, and the rest spread between. This is normalisation done per group — exactly the situation where split-apply-combine shines, because the reference values (min and max) differ from genre to genre.

The pattern — define a function that accepts a group, apply it, combine — is the full power of GroupBy. Vectorised built-ins cover the standard cases; apply covers everything else.

In [228]:
# split (apply) combine
# apply -> builtin function

genres.apply(min)

                              Series_Title Released_Year  Runtime      Genre  \
Genre                                                                          
Action                                 300          1924       45     Action   
Adventure            2001: A Space Odyssey          1925       88  Adventure   
Animation                            Akira          1940       71  Animation   
Biography                 12 Years a Slave          1928       93  Biography   
Comedy                (500) Days of Summer          1921       68     Comedy   
Crime                         12 Angry Men          1931       80      Crime   
Drama                                 1917          1925       64      Drama   
Family          E.T. the Extra-Terrestrial          1971      100     Family   
Fantasy       Das Cabinet des Dr. Caligari          1920       76    Fantasy   
Film-Noir                Shadow of a Doubt          1941      100  Film-Noir   
Horror                               Ali

In [235]:
# find number of movies starting with A for each group

def foo(group):
  return group['Series_Title'].str.startswith('A').sum()
  

In [236]:
genres.apply(foo)

Genre
Action       10
Adventure     2
Animation     2
Biography     9
Comedy       14
Crime         4
Drama        21
Family        0
Fantasy       0
Film-Noir     0
Horror        1
Mystery       0
Thriller      0
Western       0
dtype: int64

In [238]:
# find ranking of each movie in the group according to IMDB score

def rank_movie(group):
  group['genre_rank'] = group['IMDB_Rating'].rank(ascending=False)
  return group

In [240]:
genres.apply(rank_movie)

                 Series_Title Released_Year  Runtime   Genre  IMDB_Rating  \
0    The Shawshank Redemption          1994      142   Drama          9.3   
1               The Godfather          1972      175   Crime          9.2   
2             The Dark Knight          2008      152  Action          9.0   
3      The Godfather: Part II          1974      202   Crime          9.0   
4                12 Angry Men          1957       96   Crime          9.0   
..                        ...           ...      ...     ...          ...   
995    Breakfast at Tiffany's          1961      115  Comedy          7.6   
996                     Giant          1956      201   Drama          7.6   
997     From Here to Eternity          1953      118   Drama          7.6   
998                  Lifeboat          1944       97   Drama          7.6   
999              The 39 Steps          1935       86   Crime          7.6   

                 Director              Star1  No_of_Votes        Gross  \
0

In [242]:
# find normalized IMDB rating group wise

def normal(group):
  group['norm_rating'] = (group['IMDB_Rating'] - group['IMDB_Rating'].min())/(group['IMDB_Rating'].max() - group['IMDB_Rating'].min())
  return group

genres.apply(normal)

                 Series_Title Released_Year  Runtime   Genre  IMDB_Rating  \
0    The Shawshank Redemption          1994      142   Drama          9.3   
1               The Godfather          1972      175   Crime          9.2   
2             The Dark Knight          2008      152  Action          9.0   
3      The Godfather: Part II          1974      202   Crime          9.0   
4                12 Angry Men          1957       96   Crime          9.0   
..                        ...           ...      ...     ...          ...   
995    Breakfast at Tiffany's          1961      115  Comedy          7.6   
996                     Giant          1956      201   Drama          7.6   
997     From Here to Eternity          1953      118   Drama          7.6   
998                  Lifeboat          1944       97   Drama          7.6   
999              The 39 Steps          1935       86   Crime          7.6   

                 Director              Star1  No_of_Votes        Gross  \
0

# 8. GroupBy on Multiple Columns

## 8.1 Two keys, nested groups

Nothing limits you to one grouping column. Pass a list and pandas groups by their combination.

```python
duo = movies.groupby(['Director', 'Star1'])
```

Now every pair (director, lead actor) is one group. 'Christopher Nolan + Leonardo DiCaprio' is a group, 'Christopher Nolan + Matthew McConaughey' is another; the group key becomes a TUPLE, and the result of any aggregation gets a MultiIndex on those two levels — usually named after the grouped columns.

```python
duo['Gross'].sum().sort_values(ascending=False)
```

`duo['Gross'].sum()` sums the box-office revenue for every director-actor pair, and the sort answers the natural follow-up: which pairing earned the most? The output shows Akira Kurosawa / Toshirô Mifune heading the list with roughly $3 billion of combined gross — a legendary partnership quantified. The `.head(1)` variant keeps exactly the winner.

Aggregations and attributes work with tuples just as with single keys: `duo.size()` counts movies per pair, `duo.get_group(('Aamir Khan', 'Amole Gupte'))` pulls the one Taare Zameen Par row, and `duo.agg(['min', 'max', 'mean'])` reports all three statistics for every pair. Because the result is a MultiIndex, retrieving a particular pair later uses the tuple in `.loc`, like `result.loc[('Aamir Khan', 'Amole Gupte')]`.

## 8.2 Deep dive — cross-column questions

Grouping on multiple columns answers questions a single key cannot.

```python
movies.groupby(['Star1', 'Genre'])['Metascore'].mean().reset_index().sort_values('Metascore', ascending=False).head(1)
```

This chain asks for the actor-genre combination with the best average Metascore. Broken down: group by (Star1, Genre); take the mean Metascore within each pair; `reset_index()` flattens the MultiIndex back into plain columns; sort descending; keep the top row — Ellar Coltrane / Drama at a perfect 100. The reset_index call is the key habit: it turns an indexed Series into a normal DataFrame so sorting and printing behave like a flat table.

```python
duo.agg(['min', 'max', 'mean'])
```

The multi-key frame can also be aggregated broadly. `duo.agg(['min', 'max', 'mean'])` produces a DataFrame with a THREE-level column structure — director / actor on the rows, statistic function under each column — manageable at a glance and perfect for scanning extremes across every pair.

Two practical notes. First, group keys are sorted alphabetically in the result index unless you say otherwise. Second, grouping on more columns creates finer and finer groups: the more keys you add, the fewer movies usually land in each group, so results get noisier. Use one key for broad summaries, two for cross-section analytics like the director-actor combos here, and more only when the data is dense.

In [245]:
# groupby on multiple cols
duo = movies.groupby(['Director','Star1'])
duo
# size
duo.size()
# get_group
duo.get_group(('Aamir Khan','Amole Gupte'))

        Series_Title Released_Year  Runtime  Genre  IMDB_Rating    Director  \
65  Taare Zameen Par          2007      165  Drama          8.4  Aamir Khan   

          Star1  No_of_Votes      Gross  Metascore  
65  Amole Gupte       168895  1223869.0        NaN  

In [248]:
# find the most earning actor->director combo
duo['Gross'].sum().sort_values(ascending=False).head(1)

Director        Star1         
Akira Kurosawa  Toshirô Mifune    2.999877e+09
Name: Gross, dtype: float64

In [252]:
# find the best(in-terms of metascore(avg)) actor->genre combo
movies.groupby(['Star1','Genre'])['Metascore'].mean().reset_index().sort_values('Metascore',ascending=False).head(1)

              Star1  Genre  Metascore
230  Ellar Coltrane  Drama      100.0

In [253]:
# agg on multiple groupby
duo.agg(['min','max','mean'])

                                   Runtime             IMDB_Rating            \
                                       min  max   mean         min  max mean   
Director            Star1                                                      
Aamir Khan          Amole Gupte        165  165  165.0         8.4  8.4  8.4   
Aaron Sorkin        Eddie Redmayne     129  129  129.0         7.8  7.8  7.8   
Abdellatif Kechiche Léa Seydoux        180  180  180.0         7.7  7.7  7.7   
Abhishek Chaubey    Shahid Kapoor      148  148  148.0         7.8  7.8  7.8   
Abhishek Kapoor     Amit Sadh          130  130  130.0         7.7  7.7  7.7   
...                                    ...  ...    ...         ...  ...  ...   
Zaza Urushadze      Lembit Ulfsak       87   87   87.0         8.2  8.2  8.2   
Zoya Akhtar         Hrithik Roshan     155  155  155.0         8.1  8.1  8.1   
                    Vijay Varma        154  154  154.0         8.0  8.0  8.0   
Çagan Irmak         Çetin Tekindor     1

# 9. Exercise — Real Cricket Analysis

## 9.1 The deliveries dataset

The session closes by moving from movies to cricket — a ball-by-ball log of IPL deliveries.

```python
ipl = pd.read_csv('/content/deliveries.csv')
ipl.head()
```

Every row of deliveries.csv is a single ball bowled: `match_id` identifies the game, `inning` says whether it is the first or second innings, `batting_team` and `bowling_team` name the sides, `over` and `ball` count position in the innings, `batsman`, `non_striker`, and `bowler` name the three players involved, and `batsman_runs` records the runs scored off that delivery. `ipl.shape` reports (179078, 21) — over 179 thousand balls across 21 columns, a comfortably big dataset for groupby to chew on.

The questions that follow are all the same shape: group players, aggregate their column, sort, and read off the top. These are not toy queries — they are the exact analysis real cricket dashboards run:

- Top 10 batsmen by total runs — groupby batsman, sum batsman_runs, sort descending.
- Most sixes — filter to balls with `batsman_runs == 6` first, then count per batsman.
- Best at the death — filter to the last overs, then filter fours and sixes, then count.
- A batsman's record against every team — group the batsman's rows by bowling_team.

Before solving the cells, predict each answer on paper; the outputs at the bottom are the sanity check. The solutions also build a reusable `highest(batsman)` function — the course's standard move of turning a one-off query into a function.

## 9.2 Getting the data in

The two cells that follow load the deliveries file and confirm its size.

```python
ipl = pd.read_csv('/content/deliveries.csv')
ipl.head()
```

Read the CSV into a DataFrame named `ipl`, then show the first five rows — the head tells you the exact column spellings you will group and filter on: `batsman`, `bowler`, `batsman_runs`, `over`, `inning`, `match_id`.

Then:

```python
ipl.shape
```

returns `(179078, 21)`: 179,078 deliveries across 21 columns. Keep two columns in mind for the questions ahead — `batsman_runs` (the runs off each ball, the value you will sum) and `over` (which over the ball was bowled, the death-over filter).

> Tip: re-run `ipl.head()` now and then; whenever a GroupBy output looks odd, dip back into the raw rows to sanity-check the grouping keys.

In [254]:
ipl = pd.read_csv('/content/deliveries.csv')
ipl.head()

   match_id  inning         batting_team                 bowling_team  over  \
0         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
1         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
2         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
3         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
4         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   

   ball    batsman non_striker    bowler  is_super_over  ...  bye_runs  \
0     1  DA Warner    S Dhawan  TS Mills              0  ...         0   
1     2  DA Warner    S Dhawan  TS Mills              0  ...         0   
2     3  DA Warner    S Dhawan  TS Mills              0  ...         0   
3     4  DA Warner    S Dhawan  TS Mills              0  ...         0   
4     5  DA Warner    S Dhawan  TS Mills              0  ...         0   

   legbye_runs  noball_runs  penalty_runs  batsman_runs  extra_runs  \
0        

In [255]:
ipl.shape

(179078, 21)

## 9.3 Deep dive — the exercise solutions

The solution cells exercise every pattern from this session, so trace each one.

```python
ipl.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False).head(10)
```

Top ten run-getters — already seen, and the output numbers double as trivia: V Kohli 5434 and SK Raina 5415, separated by nineteen runs.

```python
six = ipl[ipl['batsman_runs'] == 6]
six.groupby('batsman')['batsman'].count().sort_values(ascending=False).head(1).index[0]
```

Sixes leader. Notice the refinement compared with the earlier code: `.index[0]` extracts just the winning batsman's name instead of printing the whole sorted Series — Chris Gayle.

```python
temp_df = ipl[ipl['over'] > 15]
temp_df = temp_df[(temp_df['batsman_runs'] == 4) | (temp_df['batsman_runs'] == 6)]
temp_df.groupby('batsman')['batsman'].count().sort_values(ascending=False).head(1).index[0]
```

Death-over boundaries: filter to overs 16+, keep fours and sixes via `|`, count per batsman — MS Dhoni.

```python
temp_df = ipl[ipl['batsman'] == 'V Kohli']
temp_df.groupby('bowling_team')['batsman_runs'].sum().reset_index()
```

A single player against every bowling team. The groupby is by bowling_team; summing runs per opponent and calling reset_index flattens the MultiIndex into a tidy two-column table of his record team by team. (Chennai Super Kings and Delhi Daredevils sit near the top.)

```python
def highest(batsman):
    temp_df = ipl[ipl['batsman'] == batsman]
    return temp_df.groupby('match_id')['batsman_runs'].sum().sort_values(ascending=False).head(1).values[0]
highest('DA Warner')
```

The reusable function: filter to the batsman's rows, sum runs PER MATCH, take the maximum single-match score. `highest('DA Warner')` returns 126 — one function, one pattern.

In [259]:
# find the top 10 batsman in terms of runs
ipl.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False).head(10)

batsman
V Kohli           5434
SK Raina          5415
RG Sharma         4914
DA Warner         4741
S Dhawan          4632
CH Gayle          4560
MS Dhoni          4477
RV Uthappa        4446
AB de Villiers    4428
G Gambhir         4223
Name: batsman_runs, dtype: int64

In [265]:
# find the batsman with max no of sixes
six = ipl[ipl['batsman_runs'] == 6]

six.groupby('batsman')['batsman'].count().sort_values(ascending=False).head(1).index[0]

'CH Gayle'

In [270]:
# find batsman with most number of 4's and 6's in last 5 overs
temp_df = ipl[ipl['over'] > 15]
temp_df = temp_df[(temp_df['batsman_runs'] == 4) | (temp_df['batsman_runs'] == 6)]
temp_df.groupby('batsman')['batsman'].count().sort_values(ascending=False).head(1).index[0]

'MS Dhoni'

In [273]:
# find V Kohli's record against all teams
temp_df = ipl[ipl['batsman'] == 'V Kohli']

temp_df.groupby('bowling_team')['batsman_runs'].sum().reset_index()

               bowling_team  batsman_runs
0       Chennai Super Kings           749
1           Deccan Chargers           306
2            Delhi Capitals            66
3          Delhi Daredevils           763
4             Gujarat Lions           283
5           Kings XI Punjab           636
6      Kochi Tuskers Kerala            50
7     Kolkata Knight Riders           675
8            Mumbai Indians           628
9             Pune Warriors           128
10         Rajasthan Royals           370
11   Rising Pune Supergiant            83
12  Rising Pune Supergiants           188
13      Sunrisers Hyderabad           509

In [279]:
# Create a function that can return the highest score of any batsman
temp_df = ipl[ipl['batsman'] == 'V Kohli']
temp_df.groupby('match_id')['batsman_runs'].sum().sort_values(ascending=False).head(1).values[0]

113

In [280]:
def highest(batsman):
  temp_df = ipl[ipl['batsman'] == batsman]
  return temp_df.groupby('match_id')['batsman_runs'].sum().sort_values(ascending=False).head(1).values[0]


In [284]:
highest('DA Warner')

126

## Summary

This session turned GroupBy — split-apply-combine — into a tool you can reach for without thinking. You split a DataFrame by one column with `movies.groupby('Genre')` and met the lazy GroupBy object, then applied the built-in aggregations sum, mean, std, and count, chained with sort_values and head to build leaderboards. Attributes and methods such as len, size, nunique, nth, get_group, groups, describe, and sample let you inspect the grouping structure itself. The agg method ran many aggregations in one call using dictionary and list syntax, and looping unpacked (group_name, group_dataframe) pairs for manual work. apply ran custom functions per group — counting titles, ranking movies inside their genre, and normalising ratings. Grouping on multiple columns produced tuple keys and MultiIndex results, answering director-actor and actor-genre questions. The cricket exercise tied it all together: top run-scorers, sixes, death-over hitters, per-opponent records, and a reusable highest-score function. GroupBy is now your default answer for every 'how many / top N / average per category' question.